# FPL: does fixture difficulty change attacker points?

**Research question.** Are individual FPL points for midfielders/forwards lower against stronger opposition, and does the effect differ for high-performing players?

We test observational associations, **not** the profitability of transfers yet. No result is included until you execute the notebook with archived data. This notebook uses the companion `fpl_fixture_hypothesis.py` module.

## 1. Historical data and pre-match feature engineering

Seasons 2022–23, 2023–24, 2024–25. We use historical community-maintained player-fixture CSVs rather than scraping the current FPL website. The script builds match ratings using only earlier results. Early matches (GW1–8) are excluded while Elo stabilizes. The **recent player quality** variable uses up to eight previous appearances of at least 60 minutes; no current-match data is used to calculate this feature.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fpl_fixture_hypothesis import get_fixture_data, model_df, fixed_effects_fit, marginal_slope, save_analysis

players, fixtures = get_fixture_data(("2022-23", "2023-24", "2024-25"), Path("data"))
print("Attacker-fixture rows:", len(players), "Completed fixtures:", len(fixtures))
print(players[["season", "name", "position", "kickoff_time", "minutes", "total_points", "prior_8_points", "own_elo", "opp_elo"]].head())

## 2. Analysis population

Primary analysis conditions on playing 60+ minutes and at least five previous 60+ minute appearances. **This is not yet a pre-deadline selection model:** we are intentionally studying output conditional on substantial playing time.

In [ ]:
d = model_df(players, min_minutes=60)
print(f"Eligible observations: {len(d):,}; unique player-seasons: {d.player_key.nunique():,}; unique fixtures: {d.fixture_key.nunique():,}")
print(d.groupby(["position", "opp_tier"], observed=True)["total_points"].agg(["size", "mean", "median"]))

## 3. Explore descriptive comparisons

For two relevant matchups, compare strong team vs strong opponent with medium team vs weak opponent. These groups differ in player cost, role, and ability; this is a **descriptive comparison** only.

In [ ]:
matchups = (d.groupby(["own_tier", "opp_tier", "position"], observed=True)
              ["total_points"].agg(["size", "mean"]).round(2))
display(matchups)
comparison = matchups.reindex(pd.MultiIndex.from_tuples([
    ('Strong','Strong','MID'), ('Medium','Weak','MID'),
    ('Strong','Strong','FWD'), ('Medium','Weak','FWD')], names=matchups.index.names))
display(comparison)

In [ ]:
quality_summary = d.groupby(["position", "quality_class", "opp_tier"], observed=True)["total_points"].agg(["size", "mean"]).reset_index()
display(quality_summary)
fig, ax = plt.subplots(figsize=(9,5))
for label in ["Bottom quartile", "Middle", "Top quartile"]:
    sample = d[d.quality_class.eq(label)]
    means = sample.groupby('opp_tier', observed=True)['total_points'].mean().reindex(["Weak","Medium","Strong"])
    ax.plot(means.index.astype(str), means, marker='o', label=label)
ax.set(xlabel='Opponent pre-match Elo tier', ylabel='Mean FPL points per 60+ minute appearance', title='Raw matchup averages, before controlling for individual players')
ax.legend()
plt.show()

## 4. Test H1 and H2 with player-season fixed effects

The within-player-season regression includes opponent Elo, own team's Elo, home advantage, earlier player form, an interaction between earlier form and opponent Elo, and a week trend. Two-way clustered standard errors (by player-season and by fixture) are used to quantify uncertainty.

- **H1**: a negative coefficient on `opp_100` indicates fewer points against opponents rated 100 Elo higher **for a player at average previous form**.
- **H2**: a positive coefficient on `quality_x_opp` means stronger players incur a smaller opponent-strength penalty. However, the stronger claim that elite players *improve* against strong opponents requires the **top-quartile marginal slope** to be positive, with uncertainty excluding zero.

In [ ]:
fit, d_model = fixed_effects_fit(d)
print(fit.summary())
q25, q75 = d_model.prior_8_points.quantile([.25, .75])
center = d_model.prior_8_points.mean()
print('Bottom quartile opponent slope:', marginal_slope(fit, q25, center))
print('Top quartile opponent slope:', marginal_slope(fit, q75, center))

## 5. Sensitivity and saved outputs

A second regression only includes players who played at least 75 minutes. If the signs or magnitudes change dramatically, the initial effect may depend on playing-time selection.

In [ ]:
d75 = model_df(players, min_minutes=75)
fit75, _ = fixed_effects_fit(d75)
print('60+ mins:', fit.params[['opp_100','quality_x_opp']].round(3).to_dict())
print('75+ mins:', fit75.params[['opp_100','quality_x_opp']].round(3).to_dict())
save_analysis(d_model, fit, Path('results'), fit75)

## Next steps

1. Validate sensitivity to Elo initialization, the exact quality definition, and whether the relationship is nonlinear.
2. Separate MID and FWD effects and evaluate clean-sheet/goal-involvement outcomes.
3. Only if fixture effects have meaningful predictive value, build an **out-of-sample expected-points model**. That model must predict minutes and account for blanks/double gameweeks.
4. Then compare transfer policies under budget, captaincy and transfer-penalty constraints.

**Warning:** Statistical significance alone is not sufficient for profitable transfers. A small but precisely estimated fixture effect may be worth less than four points.